I want to do a varaiety of extractions of cliamte data from ERA5 so that I can get climate normals, and long time series. I am hoping dask can make is less paiful. I am following the examples here: https://projectpythia.org/ERA5_interactive-cookbook/




In [1]:
import glob
import re
import matplotlib as plt
import numpy as np
import scipy as sp
import xarray as xr
import intake
import intake_esm
import pandas as pd

/srv/conda/envs/notebook/lib/python3.12/site-packages/intake_esm/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
! pip install dask_jobqueue

In [4]:
import dask
from dask.distributed import Client, performance_report,LocalCluster
from dask_jobqueue import PBSCluster

In [5]:
######## File paths ################
gdex_scratch   = "/lustre/desc1/scratch/harshah"
era5_catalog = 'https://data.gdex.ucar.edu/special_projects/pythia_2024/pythia_intake_catalogs/era5_catalog.json'
#annual_means      = '/gdex/data/special_projects/pythia_2024/annual_means/' #Posix path, only used for writing data 
annual_means_https= 'https://data.gdex.ucar.edu/special_projects/pythia_2024/annual_means/'

print(era5_catalog)

https://data.gdex.ucar.edu/special_projects/pythia_2024/pythia_intake_catalogs/era5_catalog.json


In [ ]:
USE_PBS_SCHEDULER = False

USE_DASK_GATEWAY = True # I think Dask gateway is enabled on Veda

In [6]:
def get_gateway_cluster():
    """ Create cluster through dask_gateway
    """
    from dask_gateway import Gateway

    gateway = Gateway()
    cluster = gateway.new_cluster()
    cluster.adapt(minimum=2, maximum=4)
    return cluster

In [8]:
# Connect to 

cluster = get_gateway_cluster()
from distributed import Client
client = Client(cluster)

2026-03-10 16:44:11,118 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [9]:
cluster

In [10]:
era5_cat = intake.open_esm_datastore(era5_catalog)
era5_cat

/srv/conda/envs/notebook/lib/python3.12/site-packages/intake_esm/cat.py:251: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,unique
era_id,1
datatype,2
level_type,1
step_type,7
table_code,4
param_code,164
variable,212
long_name,212
units,33
year,85


In [13]:
#era5_cat.df[['long_name','variable', 'units']].drop_duplicates().head()

,long_name,variable,units
0,Potential vorticity,PV,K m**2 kg**-1 s**-1
31,Specific rain water content,CRWC,kg kg**-1
62,Specific snow water content,CSWC,kg kg**-1
93,Geopotential,Z,m**2 s**-2
124,Temperature,T,K


Exception in callback None()
handle: <Handle cancelled>
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/tornado/iostream.py", line 1363, in _do_ssl_handshake
    self.socket.do_handshake()
  File "/srv/conda/envs/notebook/lib/python3.12/ssl.py", line 1319, in do_handshake
    self._sslobj.do_handshake()
ssl.SSLCertVerificationError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1010)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 208, in _handle_events
    handler_func(fileobj, events)
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/tornado/iostream.py", line 691, in _handle_events
    self._h

# This seems like it's only for specific pre-generated tempurature data from this one place. Now I am referencing this: 
https://era-explorer.climate.copernicus.eu/notebooks/content/precipitation/precip_monthly_climatology.html

In [15]:
! pip install earthkit.data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [earthkit.data]0m [earthkit.data]


In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import calendar
import earthkit.data

In [17]:
## per lat lon 
lat = 50.86
lng = 4.

variable = "total_precipitation"
date_range = ["1991-01-01", "2020-12-31"]

In [18]:
def retrieve_data(variable, date_range, lat, lng):
    # Define the dataset and request parameters
    dataset = "reanalysis-era5-single-levels-timeseries"
    request = {
        "variable": [
        variable,  # Variable to retrieve
        ],
        "date": date_range,  # Date range for the data
        "location": {"longitude": lng, "latitude": lat},  # Location coordinates
        "data_format": "netcdf"  # Format of the retrieved data
    }

    # Use "earthkit" to retrieve the data
    ekds = earthkit.data.from_source(
        "cds", dataset, request
    ).to_xarray()

    return ekds

In [19]:
data = retrieve_data(variable, date_range, lat, lng)

2026-03-10 18:07:14,179 INFO [2026-02-16T00:00:00] - To generate this ERA5 hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/R6cfHg) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202).
2026-03-10 18:07:14,180 INFO Request ID is 8c35b342-0a5a-4347-87f8-43ed6f28b7eb
2026-03-10 18:07:14,391 INF

f98f7ee7994a8e71402be3ec8a6f5e64.zip:   0%|          | 0.00/3.02M [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
# Make a function to compute the monthly precipitation climatology
def precipMonthlyClimatology():
    """
    Calculate the monthly climatology of precipitation.

    This function reads precipitation data from a NetCDF file,
    converts the time coordinate to a pandas datetime index,
    and then resamples the data to calculate the monthly 
    climatology. The resulting climatology is returned in millimeters.

    Returns:
        pandas.DataFrame: A DataFrame containing the monthly climatology
        of precipitation in millimeters, indexed by month.
    """


    data_tp_pt = data.tp

    # Convert the time coordinate to a pandas datetime index
    time_index = pd.to_datetime(data_tp_pt.valid_time.values)

    # Create a DataFrame for easier manipulation
    df = pd.DataFrame(data_tp_pt.values, index=time_index, columns=['tp'])

    df_monthly = df.resample('d').sum()
    df_monthly['month_day'] = df_monthly.index.month.astype("str") + "_" + df_monthly.index.day.astype("str")
    monthly_climatology = df_monthly.groupby('month_day').mean() * 1000

    # Get the actual lat/lon used
    nearest_lat = data_tp_pt.latitude.values
    nearest_lng = data_tp_pt.longitude.values

    return monthly_climatology, nearest_lat, nearest_lng


# Call our function
clim, nearest_lat, nearest_lng = precipMonthlyClimatology()